# Data Patching 

# Load Data & Configure Local Ollama

In [2]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datasets import load_dataset, Dataset
from huggingface_hub import login
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Login to Hugging Face (Paste token if asked)
login()

# ---> UPDATE THIS <---
repo_id = "ShravSiddhpura/cybersec-slopsquatting-crag" 
model_name = "mistral-nemo"  # Ensure you ran `ollama run mistral-nemo` in terminal

print(f"Loading dataset from {repo_id}...")
dataset = load_dataset(repo_id, split="train")

# Create a WORKING COPY of the data to patch
all_data_patched = dataset.to_pandas().to_dict(orient="records")

print(f"Loaded {len(all_data_patched)} rows. Ready to patch locally.")

Loading dataset from ShravSiddhpura/cybersec-slopsquatting-crag...


README.md:   0%|          | 0.00/342 [00:00<?, ?B/s]

c:\Users\ADMIN\Desktop\coding shoding\llm_eng\.venv\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--ShravSiddhpura--cybersec-slopsquatting-crag. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/470k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/904 [00:00<?, ? examples/s]

Loaded 904 rows. Ready to patch locally.


In [ ]:
def aggressive_rewrite_local(prompt, rigid_answer):
    # Pointing to your local Ollama instance
    url = "http://localhost:11434/v1/chat/completions"
    headers = {"Content-Type": "application/json"}
    
    system_prompt = (
        "You are an AI cybersecurity expert. Rewrite the provided factual answer into a highly detailed, "
        "conversational response. It MUST be at least 3 paragraphs long. "
        "You MUST include the exact package name and CVE number provided. "
        "Make it sound exactly like a verbose AI assistant answering a complex coding question."
    )
    
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Prompt: {prompt}\nFactual Answer: {rigid_answer}"}
        ],
        "temperature": 0.7
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        data = response.json()
        if "error" in data:
            print(f"Error: {data['error']}")
            return None
        return data["choices"][0]["message"]["content"]
    except Exception as e:
        print(f"Local API Error: {e}")
        return None

# Loop and Patch
update_count = 0
print(f"Patching dataset using local {model_name}...")

for item in tqdm(all_data_patched):
    # Target Grounded (0) answers that are suspiciously short (e.g. under 600 chars)
    if item['label'] == 0 and len(str(item['answer'])) < 600: 
        new_answer = aggressive_rewrite_local(item['prompt'], item['answer'])
        
        if new_answer:
            item['answer'] = new_answer
            update_count += 1

print(f"\nSuccessfully rewrote {update_count} rows in memory.")

Patching dataset using local mistral-nemo...


  3%|▎         | 29/904 [09:15<7:56:50, 32.70s/it]